# 서울 지하철 승객 특성 및 호선별 민원 데이터 심층 분석 프로젝트

### 프로젝트 개요
- **분석 목적**:
  1. 2023년 서울 지하철 1~8호선 수송 실적과 민원 접수 데이터를 연계 분석
  2. 노선별 민원 발생량 및 인구 대비 발생률(10만 명당 건수) 비교 검정
  3. 노선별 민원 유형(냉난방, 질서저해, 이용예절 등) 구성비 차이 검정
  4. 승객 연령 대리변수(노인 우대권, 청소년, 성인 일반 등)와 민원 간의 상관관계 규명

## 1. 환경 설정 및 라이브러리 임포트
- 데이터 분석(`pandas`, `numpy`), 통계 검정(`scipy.stats`), 시각화(`matplotlib`, `seaborn`) 라이브러리를 설정합니다.
- 한글 깨짐 방지를 위한 윈도우 기본 한글 폰트(`Malgun Gothic`)를 설정합니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager

# 한글 폰트 설정
installed_fonts = {f.name for f in font_manager.fontManager.ttflist}
font_name = next((f for f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR'] if f in installed_fonts), None)
if font_name:
    plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = Path('.').resolve()
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

print('환경 설정 완료! 데이터 경로:', BASE_DIR)

## 2. 데이터 로드 및 전처리 (메모리 집계)
- **승객 데이터**: 상/하반기 CSV 2개(`subway_passenger_2023_1.csv`, `subway_passenger_2023_2.csv`)를 청크 단위로 읽어 '승차' 기준 승객 수와 연령그룹(어린이, 청소년, 성인 일반, 노인 우대권)을 집계합니다.
- **민원 데이터**: `subway_complaint_2023.csv`를 읽어 호선별/민원구분별 피벗 테이블을 생성합니다.
- 두 데이터를 `호선` 기준으로 결합하여 마스터 데이터프레임(`df_master`)을 구축합니다.

In [ ]:
TIME_COLS = [
    '06시간대이전', '06-07시간대', '07-08시간대', '08-09시간대', '09-10시간대',
    '10-11시간대', '11-12시간대', '12-13시간대', '13-14시간대', '14-15시간대',
    '15-16시간대', '16-17시간대', '17-18시간대', '18-19시간대', '19-20시간대',
    '20-21시간대', '21-22시간대', '22-23시간대', '23-24시간대', '24시간대이후'
]
RUSH_COLS = ['07-08시간대', '08-09시간대', '18-19시간대', '19-20시간대']
USE_COLS = ['호선명', '승하차구분', '승객유형'] + TIME_COLS

def map_passenger_type(ptype: str) -> str:
    if pd.isna(ptype):
        return '기타'
    ptype = str(ptype).strip()
    if '어린이' in ptype:
        return '어린이'          # 만 6~12세
    elif '청소년' in ptype or '중고생' in ptype:
        return '청소년'          # 만 13~18세
    elif '우대권' in ptype:
        return '노인_우대권'      # 만 65세 이상 경로우대 중심
    elif '일반' in ptype:
        return '성인_일반'        # 만 19~64세 일반 성인
    return '기타'

# 1) 대용량 승객 데이터 집계
chunks_list = []
for fname in ['subway_passenger_2023_1.csv', 'subway_passenger_2023_2.csv']:
    print(f'-> 로드 및 집계 중: {fname}...')
    for chunk in pd.read_csv(BASE_DIR / fname, encoding='cp949', usecols=USE_COLS, chunksize=150000):
        c_board = chunk[chunk['승하차구분'] == '승차'].copy()
        c_board['총승객'] = c_board[TIME_COLS].sum(axis=1)
        c_board['출퇴근승객'] = c_board[RUSH_COLS].sum(axis=1)
        c_board['호선'] = pd.to_numeric(c_board['호선명'], errors='coerce')
        c_board = c_board[c_board['호선'].isin(range(1, 9))].copy()
        c_board['호선'] = c_board['호선'].astype(int)
        c_board['연령그룹'] = c_board['승객유형'].apply(map_passenger_type)
        agg = c_board.groupby(['호선', '연령그룹']).agg(총승객=('총승객', 'sum'), 출퇴근승객=('출퇴근승객', 'sum')).reset_index()
        chunks_list.append(agg)

summary = pd.concat(chunks_list, ignore_index=True).groupby(['호선', '연령그룹']).sum().reset_index()
df_passengers = summary.pivot_table(index='호선', columns='연령그룹', values='총승객', fill_value=0).reset_index()
rush_sum = summary.groupby('호선')['출퇴근승객'].sum().reset_index()
df_passengers = pd.merge(df_passengers, rush_sum, on='호선')
df_passengers['총승객'] = df_passengers[['노인_우대권', '성인_일반', '어린이', '청소년', '기타']].sum(axis=1)

# 비율 컬럼 산출
df_passengers['우대권_비율(%)'] = (df_passengers['노인_우대권'] / df_passengers['총승객']) * 100
df_passengers['청소년_비율(%)'] = (df_passengers['청소년'] / df_passengers['총승객']) * 100
df_passengers['성인일반_비율(%)'] = (df_passengers['성인_일반'] / df_passengers['총승객']) * 100
df_passengers['어린이_비율(%)'] = (df_passengers['어린이'] / df_passengers['총승객']) * 100
df_passengers['출퇴근혼잡비율(%)'] = (df_passengers['출퇴근승객'] / df_passengers['총승객']) * 100

# 2) 민원 데이터 로드 및 피벗
complaint_raw = pd.read_csv(BASE_DIR / 'subway_complaint_2023.csv', encoding='cp949')
complaint_clean = complaint_raw[complaint_raw['호선'] != '기타'].copy()
complaint_clean['호선'] = complaint_clean['호선'].astype(int)
df_complaints = complaint_clean.pivot_table(index='호선', columns='민원구분', values='민원접수건수', fill_value=0).reset_index()
df_complaints.rename(columns={'기타': '기타민원'}, inplace=True)
df_complaints['총민원건수'] = df_complaints[['전동차 냉난방', '열차내 질서저해', '열차이용예절', '기타민원']].sum(axis=1)

# 3) 통합 마스터 DataFrame 구성
df_master = pd.merge(df_passengers, df_complaints, on='호선')
df_master['십만명당_총민원'] = (df_master['총민원건수'] / df_master['총승객']) * 100000
df_master['십만명당_냉난방'] = (df_master['전동차 냉난방'] / df_master['총승객']) * 100000
df_master['십만명당_질서저해'] = (df_master['열차내 질서저해'] / df_master['총승객']) * 100000
df_master['십만명당_이용예절'] = (df_master['열차이용예절'] / df_master['총승객']) * 100000
df_master['십만명당_기타민원'] = (df_master['기타민원'] / df_master['총승객']) * 100000

print('전처리 완료!')
df_master[['호선', '총승객', '총민원건수', '십만명당_총민원', '우대권_비율(%)', '청소년_비율(%)', '성인일반_비율(%)']]

## 3. [가설 1] 노선별 승객 대비 민원 발생률 차이 검정
- **귀무가설($H_0$)**: 호선별 승객 10만 명당 민원 발생률은 차이가 없다. (모든 노선의 승객 대비 민원 비율은 동일하다)
- **대립가설($H_1$)**: 적어도 한 개 이상의 호선에서 승객 대비 민원 발생률에 유의미한 차이가 있다.
- **검정 방법**: 카이제곱 적합도 검정 (Goodness-of-Fit Test)

In [ ]:
total_passengers = df_master['총승객'].sum()
total_complaints = df_master['총민원건수'].sum()
expected_complaints = df_master['총승객'] * (total_complaints / total_passengers)

chi1_stat, chi1_pval = stats.chisquare(f_obs=df_master['총민원건수'], f_exp=expected_complaints)
dof1 = len(df_master) - 1

print('카이제곱 통계량 =', round(chi1_stat, 2), 'p-value =', chi1_pval, '자유도 =', dof1)
if chi1_pval < 0.05:
    print('결론: 유의수준 0.05에서 p-value < 0.05이므로 귀무가설을 강력히 기각합니다.')
    print('      승객 규모를 보정하더라도 7호선(105.2건), 2호선(88.8건)의 민원율이 8호선(24.6건) 등에 비해 유의미하게 높습니다.')
else:
    print('결론: 귀무가설을 기각하지 못합니다.')

# [시각화 1] 총 민원 규모 vs 10만 명당 발생률 (위치 일치)
fig, ax1 = plt.subplots(figsize=(10, 5))
lines = [f'{int(h)}호선' for h in df_master['호선']]
x = np.arange(len(lines))

# 막대와 꺾은선 모두 x축 정중앙(x)에 배치 (오프셋 제거)
bars = ax1.bar(x, df_master['총민원건수'] / 1000, width=0.45, color='#4dabf7', edgecolor='black', alpha=0.85, label='총 민원 접수건수 (천 건)')
ax1.set_xlabel('호선', fontsize=11, fontweight='bold')
ax1.set_ylabel('총 민원 건수 (천 건)', color='#1971c2', fontsize=11, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#1971c2')
ax1.set_xticks(x)
ax1.set_xticklabels(lines, fontsize=10.5)
ax1.set_ylim(0, max(df_master['총민원건수'] / 1000) * 1.15)
ax1.grid(axis='y', linestyle=':', alpha=0.5)

for bar in bars:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 8, f'{h:.0f}천', ha='center', fontsize=9.5, fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(x, df_master['십만명당_총민원'], color='#ff6b6b', marker='o', linewidth=2.8, markersize=8.5, label='10만 명당 민원 건수')
ax2.set_ylabel('10만 명당 민원 발생률 (건)', color='#c92a2a', fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#c92a2a')
ax2.set_ylim(0, max(df_master['십만명당_총민원']) * 1.18)

for i, val in enumerate(df_master['십만명당_총민원']):
    ax2.text(x[i], val + 3.2, f'{val:.1f}건', ha='center', fontsize=9.5, fontweight='bold', color='#c92a2a')

# 범례 통합 표시
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left', frameon=True, facecolor='white', edgecolor='#dee2e6')

plt.title('2023 서울 지하철 호선별 민원 규모 및 인구 보정 발생률 비교 (가설 1)', fontsize=13, fontweight='bold', pad=15)
fig.tight_layout()
plt.savefig(RESULTS_DIR / '01_complaint_by_line_rate.png', dpi=150)
plt.show()

## 4. [가설 2] 노선별 민원 유형 구성 비율 차이 검정
- **귀무가설($H_0$)**: 노선(1~8호선)과 접수되는 민원 유형은 서로 독립이다. (노선별 민원 구성비가 동일하다)
- **대립가설($H_1$)**: 노선과 민원 유형은 서로 독립이 아니다. (노선별 특정 민원 편중이 존재한다)
- **검정 방법**: 카이제곱 독립성 검정 (분할표 8×4, 자유도 21)

In [ ]:
complaint_cols = ['전동차 냉난방', '열차내 질서저해', '열차이용예절', '기타민원']
ct_matrix = df_master.set_index('호선')[complaint_cols]
ct_matrix.columns = ['전동차 냉난방', '열차내 질서저해', '열차이용예절', '기타']

chi2_stat, chi2_pval, dof2, _ = stats.chi2_contingency(ct_matrix)
print(f"카이제곱 독립성 통계량 = {chi2_stat:,.2f}, p-value = {chi2_pval:.4e}, 자유도 = {dof2}")
if chi2_pval < 0.05:
    print("결론: 유의수준 0.05에서 p-value < 0.05이므로 귀무가설을 기각합니다.")
    print("      노선별 민원 유형 구성비에 뚜렷한 차이가 있습니다.")
    print("      (2·7호선은 냉난방이 약 80% 편중, 1호선은 질서저해(9.2%) 및 기타(29.5%) 비중이 두드러짐)")

# [시각화 2] 100% 누적 가로 막대 그래프
fig, ax = plt.subplots(figsize=(11, 5.5))
ct_ratio = ct_matrix.div(ct_matrix.sum(axis=1), axis=0) * 100
colors_type = ['#339af0', '#fcc419', '#51cf66', '#868e96']

left = np.zeros(len(lines))
for j, col_name in enumerate(ct_matrix.columns):
    vals = ct_ratio[col_name].values
    ax.barh(lines, vals, left=left, color=colors_type[j], label=col_name, edgecolor='black', alpha=0.88, height=0.6)
    for i in range(len(lines)):
        if vals[i] > 3.5:
            ax.text(left[i] + vals[i] / 2, i, f'{vals[i]:.1f}%', ha='center', va='center', color='white' if j!=1 else 'black', fontweight='bold', fontsize=9)
    left += vals

ax.set_xlim(0, 100)
ax.set_xlabel('민원 유형별 구성 비율 (%)', fontsize=11, fontweight='bold')
ax.set_title(f'호선별 민원 유형 구성비 비교 (카이제곱 독립성 검정 p < 0.001) - 가설 2', fontsize=13, fontweight='bold')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.18), ncol=4, frameon=False, fontsize=10)
ax.grid(axis='x', linestyle=':', alpha=0.5)
ax.invert_yaxis()
fig.tight_layout()
plt.savefig(RESULTS_DIR / '02_complaint_composition_ratio.png', dpi=150)
plt.show()

# [시각화 3] 카이제곱 분포 곡선 및 기각역 시각화
fig, ax = plt.subplots(figsize=(9, 4.5))
chi_crit = stats.chi2.ppf(0.95, dof2)
x_chi = np.linspace(0, max(50, chi_crit + 15), 500)
y_chi = stats.chi2.pdf(x_chi, dof2)

ax.plot(x_chi, y_chi, color='#495057', linewidth=2)
ax.fill_between(x_chi, y_chi, where=(x_chi >= chi_crit), color='#e03131', alpha=0.35, label=f'기각역 (χ² ≥ {chi_crit:.2f}, α=0.05)')
ax.set_title(f'카이제곱 분포 (자유도 {dof2}) : 관측 통계량({chi2_stat:,.0f})이 극단적 기각역에 위치', fontsize=12, fontweight='bold')
ax.set_xlabel('χ² 값', fontsize=10.5)
ax.set_ylabel('확률밀도', fontsize=10.5)
ax.annotate(f'관측된 χ² = {chi2_stat:,.1f}\n(p-value < 0.001)',
            xy=(45, 0.01), xytext=(35, 0.04),
            arrowprops=dict(facecolor='#1971c2', shrink=0.08, width=1.5, headwidth=6),
            fontsize=10.5, fontweight='bold', color='#1971c2')
ax.legend(frameon=False, loc='upper right')
ax.grid(axis='y', linestyle=':', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.savefig(RESULTS_DIR / '03_chi_square_distribution.png', dpi=150)
plt.show()

## 5. [가설 3] 승객 특성(연령대 대리변수)과 민원 발생률 간의 상관분석
- **귀무가설($H_0$)**: 노선별 승객 연령대 비율(우대권, 청소년, 성인 등)과 민원 발생률 간에는 상관관계가 없다 ($r = 0$).
- **대립가설($H_1$)**: 노선별 승객 연령대 비율과 민원 발생률 간에는 유의미한 상관관계가 있다 ($r \ne 0$).

In [ ]:
r_elder_order, p_elder_order = stats.pearsonr(df_master['우대권_비율(%)'], df_master['십만명당_질서저해'])
r_elder_temp, p_elder_temp = stats.pearsonr(df_master['우대권_비율(%)'], df_master['십만명당_냉난방'])
r_youth_manner, p_youth_manner = stats.pearsonr(df_master['청소년_비율(%)'], df_master['십만명당_이용예절'])
r_adult_temp, p_adult_temp = stats.pearsonr(df_master['성인일반_비율(%)'], df_master['십만명당_냉난방'])

print(f"1) 우대권(노인층) 비율 vs 10만 명당 질서저해: r = {r_elder_order:.3f} (p = {p_elder_order:.3f})")
print(f"2) 우대권(노인층) 비율 vs 10만 명당 냉난방:     r = {r_elder_temp:.3f} (p = {p_elder_temp:.3f})")
print(f"3) 청소년 비율 vs 10만 명당 이용예절:         r = {r_youth_manner:.3f} (p = {p_youth_manner:.3f})")
print(f"4) 성인 일반 비율 vs 10만 명당 냉난방:          r = {r_adult_temp:.3f} (p = {p_adult_temp:.3f})")

# [시각화 4] 호선별 승객 연령대 구성 비율
fig, ax = plt.subplots(figsize=(10, 5))
age_cols = ['우대권_비율(%)', '청소년_비율(%)', '성인일반_비율(%)', '어린이_비율(%)']
age_labels = ['노인 우대권 (65세 이상)', '청소년 (13~18세)', '성인 일반 (19~64세)', '어린이']
colors_age = ['#fab005', '#ff922b', '#228be6', '#20c997']
age_df = df_master.set_index('호선')[age_cols]
age_df.index = lines

left = np.zeros(len(lines))
for j, col_name in enumerate(age_cols):
    vals = age_df[col_name].values
    ax.barh(lines, vals, left=left, color=colors_age[j], label=age_labels[j], edgecolor='black', alpha=0.85, height=0.6)
    for i in range(len(lines)):
        if vals[i] > 2.0:
            ax.text(left[i] + vals[i] / 2, i, f'{vals[i]:.1f}%', ha='center', va='center', color='white' if j==2 else 'black', fontweight='bold', fontsize=8.5)
    left += vals

ax.set_xlim(0, 100)
ax.set_xlabel('승객 연령그룹별 구성비 (%)', fontsize=11, fontweight='bold')
ax.set_title('호선별 승객 연령대(대리변수) 구성 비율 비교', fontsize=13, fontweight='bold')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.18), ncol=4, frameon=False, fontsize=9.5)
ax.grid(axis='x', linestyle=':', alpha=0.5)
ax.invert_yaxis()
fig.tight_layout()
plt.savefig(RESULTS_DIR / '04_passenger_age_composition.png', dpi=150)
plt.show()

# [시각화 5] 산점도 및 선형 회귀선
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
sns.regplot(data=df_master, x='우대권_비율(%)', y='십만명당_질서저해', ax=ax1,
            color='#1971c2', scatter_kws={'s': 90, 'edgecolor': 'black'}, line_kws={'color': '#fa5252', 'linewidth': 2})
for _, row in df_master.iterrows():
    ax1.annotate(f"{int(row['호선'])}호선", (row['우대권_비율(%)'], row['십만명당_질서저해']),
                 textcoords="offset points", xytext=(0, 6), ha='center', fontsize=9, fontweight='bold')
ax1.set_title(f'우대권(노인층) 비율 vs 10만 명당 질서저해 민원\n(r = {r_elder_order:.2f}, p = {p_elder_order:.3f})', fontsize=11, fontweight='bold')
ax1.set_xlabel('우대권 탑승객 비율 (%)', fontsize=10.5)
ax1.set_ylabel('10만 명당 열차내 질서저해 민원 (건)', fontsize=10.5)
ax1.grid(True, linestyle=':', alpha=0.5)

sns.regplot(data=df_master, x='성인일반_비율(%)', y='십만명당_냉난방', ax=ax2,
            color='#20c997', scatter_kws={'s': 90, 'edgecolor': 'black'}, line_kws={'color': '#e03131', 'linewidth': 2})
for _, row in df_master.iterrows():
    ax2.annotate(f"{int(row['호선'])}호선", (row['성인일반_비율(%)'], row['십만명당_냉난방']),
                 textcoords="offset points", xytext=(0, 6), ha='center', fontsize=9, fontweight='bold')
ax2.set_title(f'성인 일반 승객 비율 vs 10만 명당 냉난방 민원\n(r = {r_adult_temp:.2f}, p = {p_adult_temp:.3f})', fontsize=11, fontweight='bold')
ax2.set_xlabel('성인 일반 탑승객 비율 (%)', fontsize=10.5)
ax2.set_ylabel('10만 명당 전동차 냉난방 민원 (건)', fontsize=10.5)
ax2.grid(True, linestyle=':', alpha=0.5)

fig.suptitle('승객 연령 특성과 주요 민원 발생률 간의 선형 회귀 분석 (가설 3)', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.savefig(RESULTS_DIR / '05_passenger_features_vs_complaints_regression.png', dpi=150)
plt.show()

# [시각화 6] 종합 상관관계 히트맵
fig, ax = plt.subplots(figsize=(9, 7))
corr_cols = [
    '우대권_비율(%)', '청소년_비율(%)', '성인일반_비율(%)', '출퇴근혼잡비율(%)',
    '십만명당_총민원', '십만명당_냉난방', '십만명당_질서저해', '십만명당_이용예절'
]
corr_labels = [
    '우대권 비율', '청소년 비율', '성인일반 비율', '출퇴근 혼잡비율',
    '10만명당 총민원', '10만명당 냉난방', '10만명당 질서저해', '10만명당 이용예절'
]
corr_matrix = df_master[corr_cols].corr()
corr_matrix.columns = corr_labels
corr_matrix.index = corr_labels

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1,
            linewidths=0.5, linecolor='gray', ax=ax, cbar_kws={'label': 'Pearson Correlation (r)'})
ax.set_title('지하철 승객 연령 특성 및 민원 발생률 종합 상관관계 히트맵', fontsize=12, fontweight='bold', pad=12)
fig.tight_layout()
plt.savefig(RESULTS_DIR / '06_comprehensive_correlation_heatmap.png', dpi=150)
plt.show()

## 6. 결론 및 요약
1. **노선별 민원율 차이**: 7호선(105.2건)과 2호선(88.8건)은 총 승객 수를 보정하더라도 8호선(24.6건), 1호선(32.1건)에 비해 통계적으로 유의미하게 민원 발생률이 높습니다 ($p < 0.001$).
2. **노선별 민원 성격**: 2호선과 7호선은 약 80%가 냉난방 민원에 집중되어 있는 반면, 1호선은 질서저해(9.2%) 및 기타 민원(29.5%)의 비중이 타 노선 대비 2~3배 높습니다 ($p < 0.001$).
3. **승객 연령 특성과의 상관관계**: 노인 우대권 비율과 질서저해 민원 발생률 간에는 유의미한 양의 상관관계가 나타나지 않았습니다 ($r = -0.300, p = 0.470$). 반면 성인 일반 승객 비율이 높은 혼잡 노선일수록 냉난방 불만이 급증하는 패턴($r = +0.697$)이 확인되었습니다.